# Build Cuisine RAG Index
### all-MiniLM-L6-v2 → GGUF + cuisine_index.bin

**Produces two files for the Android app:**
- `all-minilm-l6-v2.Q8_0.gguf` — embedding model for on-device inference
- `cuisine_index.bin` — 400 pre-embedded Q&A vectors

**Push both to the app's internal storage:**
```bash
adb push all-minilm-l6-v2.Q8_0.gguf  /data/data/com.cuisine.rag/files/
adb push cuisine_index.bin             /data/data/com.cuisine.rag/files/
adb push qwen3-0.6b-cuisine.Q4_K_M.gguf /data/data/com.cuisine.rag/files/
```

---

## 1 · Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers torch
!apt-get install -y cmake build-essential git 2>/dev/null

## 2 · Upload Qwen_cuisine_qa.txt

In [ ]:
from google.colab import files

uploaded = files.upload()  # choose Qwen_cuisine_qa.txt

with open('Qwen_cuisine_qa.txt', 'r', encoding='utf-8') as f:
    raw = f.read()

qa_pairs = []
for block in raw.strip().split('\n\n'):
    block = block.strip()
    if not block:
        continue
    lines = block.split('\n', 1)
    if len(lines) == 2 and lines[0].startswith('Q:') and lines[1].startswith('A:'):
        qa_pairs.append({
            'question': lines[0][2:].strip(),
            'answer':   lines[1][2:].strip(),
        })

print(f'Parsed {len(qa_pairs)} Q&A pairs')

## 3 · Embed all questions with all-MiniLM-L6-v2

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

questions = [p['question'] for p in qa_pairs]
print(f'Embedding {len(questions)} questions...')

# Returns shape (N, 384), L2-normalised by default
embeddings = embedder.encode(
    questions,
    normalize_embeddings = True,
    batch_size           = 64,
    show_progress_bar    = True,
)

print(f'Embeddings shape: {embeddings.shape}')  # (400, 384)
print(f'Dtype: {embeddings.dtype}')

## 4 · Write cuisine_index.bin

In [ ]:
import struct

MAGIC   = 0x43555349   # 'CUSI'
VERSION = 1
DIM     = 384

OUTPUT_PATH = 'cuisine_index.bin'

with open(OUTPUT_PATH, 'wb') as f:
    # Header: magic, version, count, dim  (all Int32 big-endian)
    f.write(struct.pack('>iiii', MAGIC, VERSION, len(qa_pairs), DIM))

    for i, pair in enumerate(qa_pairs):
        q_bytes = pair['question'].encode('utf-8')
        a_bytes = pair['answer'].encode('utf-8')

        # question length + bytes
        f.write(struct.pack('>i', len(q_bytes)))
        f.write(q_bytes)

        # answer length + bytes
        f.write(struct.pack('>i', len(a_bytes)))
        f.write(a_bytes)

        # embedding: 384 float32 little-endian
        f.write(embeddings[i].astype('<f4').tobytes())

import os
size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f'Written: {OUTPUT_PATH}  ({size_kb:.1f} KB)')

## 5 · Verify the index
Quick sanity check — search for a known question.

In [ ]:
import struct, numpy as np

def load_index(path):
    entries = []
    with open(path, 'rb') as f:
        magic, version, count, dim = struct.unpack('>iiii', f.read(16))
        assert magic == 0x43555349, 'Bad magic'
        for _ in range(count):
            qlen = struct.unpack('>i', f.read(4))[0]
            q    = f.read(qlen).decode('utf-8')
            alen = struct.unpack('>i', f.read(4))[0]
            a    = f.read(alen).decode('utf-8')
            emb  = np.frombuffer(f.read(dim * 4), dtype='<f4').copy()
            entries.append((q, a, emb))
    return entries

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

entries = load_index('cuisine_index.bin')
print(f'Loaded {len(entries)} entries')

# Search for a question
query = 'Tell me about paella'
q_emb = embedder.encode(query, normalize_embeddings=True)

scores = [(cosine(q_emb, e[2]), e[0], e[1]) for e in entries]
scores.sort(reverse=True)

print(f'\nQuery: "{query}"')
print('\nTop 3 matches:')
for score, question, answer in scores[:3]:
    print(f'  [{score:.3f}] {question}')
    print(f'          {answer[:120]}...')

## 6 · Convert all-MiniLM-L6-v2 to GGUF

In [ ]:
import subprocess

LLAMA_DIR = '/root/.unsloth/llama.cpp'

# Check if llama.cpp is already present (from GGUF export notebook)
import os
if not os.path.exists(LLAMA_DIR):
    print('Cloning llama.cpp...')
    subprocess.run([
        'git', 'clone', '--depth=1',
        'https://github.com/ggerganov/llama.cpp',
        LLAMA_DIR,
    ], check=True)
else:
    print('llama.cpp already present')

# Install Python deps for the converter
subprocess.run(['pip', 'install', '--quiet', 'gguf', 'sentencepiece', 'protobuf'], check=True)
print('Ready')

In [ ]:
# Save the HuggingFace model to disk so llama.cpp can convert it
MODEL_HF_DIR  = './all-minilm-l6-v2-hf'
GGUF_BF16     = './all-minilm-l6-v2.BF16.gguf'
GGUF_Q8       = './all-minilm-l6-v2.Q8_0.gguf'

print('Saving HF model to disk...')
embedder.save(MODEL_HF_DIR)
print(f'Saved to {MODEL_HF_DIR}')

In [ ]:
# Convert HF → BF16 GGUF
print('Converting to GGUF BF16...')
result = subprocess.run([
    'python3', f'{LLAMA_DIR}/convert_hf_to_gguf.py',
    MODEL_HF_DIR,
    '--outfile', GGUF_BF16,
    '--outtype', 'bf16',
], capture_output=True, text=True)

if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Conversion failed')

print(f'BF16 GGUF: {os.path.getsize(GGUF_BF16)/1e6:.1f} MB')

In [ ]:
# Build llama-quantize and quantise to Q8_0
print('Building llama-quantize...')
subprocess.run([
    'cmake', '-B', f'{LLAMA_DIR}/build',
    '-DGGML_CUDA=OFF', '-DCMAKE_BUILD_TYPE=Release',
    '-S', LLAMA_DIR,
], check=True, capture_output=True)

subprocess.run([
    'cmake', '--build', f'{LLAMA_DIR}/build',
    '--target', 'llama-quantize', '-j4',
], check=True, capture_output=True)

print('Quantising to Q8_0...')
subprocess.run([
    f'{LLAMA_DIR}/build/bin/llama-quantize',
    GGUF_BF16, GGUF_Q8, 'Q8_0',
], check=True)

print(f'Q8_0 GGUF: {os.path.getsize(GGUF_Q8)/1e6:.1f} MB')

## 7 · Download both files

In [ ]:
from google.colab import files

print('Downloading cuisine_index.bin...')
files.download('cuisine_index.bin')

print('Downloading all-minilm-l6-v2.Q8_0.gguf...')
files.download(GGUF_Q8)

print()
print('Push both to Android with:')
print('  adb push cuisine_index.bin             /data/data/com.cuisine.rag/files/')
print('  adb push all-minilm-l6-v2.Q8_0.gguf   /data/data/com.cuisine.rag/files/')
print('  adb push qwen3-0.6b-cuisine.Q4_K_M.gguf /data/data/com.cuisine.rag/files/')

---
## Notes

### Index binary format
The `cuisine_index.bin` format is deliberately simple — no external library needed to read it on Android.
The Android `VectorIndex.kt` reads it with a `DataInputStream` directly.

### Why Q8_0 for the embedding model?
The embedding model is tiny (22MB at Q8_0) and accuracy matters — we want embeddings as close to
the original float32 values as possible. Q4 would save ~10MB but risks degrading similarity scores.

### Updating the index
To add more Q&A pairs: append to `Qwen_cuisine_qa.txt`, re-run from cell 2.
The index file is regenerated in seconds — no model retraining needed.

### JNI implementation note
The `LlamaCppEngine` JNI bridge in the Android project declares the native methods.
You need to implement the corresponding C++ file (`llama_jni.cpp`) using the
llama.cpp C API. The prebuilt `libllama.so` for Android arm64-v8a can be obtained from:
  https://github.com/ggerganov/llama.cpp/releases
or built from source with the Android NDK:
  cmake -DCMAKE_TOOLCHAIN_FILE=$NDK/build/cmake/android.toolchain.cmake \\
        -DANDROID_ABI=arm64-v8a -DANDROID_PLATFORM=android-26 \\
        -DGGML_OPENMP=OFF -B build-android
  cmake --build build-android --target llama -j4